In [1]:
from datasets import load_dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import torch
from torch.utils.data import WeightedRandomSampler
import numpy as np

# Load the dataset
dataset = load_dataset("ailsntua/QEvasion")

# Prepare labels
labels = dataset["train"].unique("clarity_label")
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}


def add_labels(example):
    example["labels"] = label2id[example["clarity_label"]]
    return example


dataset = dataset.map(add_labels)
dataset = dataset.remove_columns(
    [
        col
        for col in dataset["train"].column_names
        if col not in ["question", "interview_answer", "labels"]
    ]
)

print("Dataset ready:")
print(dataset)
print(f"Labels mapped: {label2id}")

from transformers import AutoTokenizer

model_checkpoint = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


def tokenize_function(examples):
    return tokenizer(
        examples["question"],
        examples["interview_answer"],
        truncation=True,
        padding="max_length",
        max_length=2048,  # Reduced from 4096 to avoid memory issues
    )


tokenized_datasets = dataset.map(tokenize_function, batched=True)

from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id
)


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)

    # Use macro averaging for balanced metrics across classes
    precision_macro = precision_score(labels, predictions, average="macro")
    recall_macro = recall_score(labels, predictions, average="macro")
    f1_macro = f1_score(labels, predictions, average="macro")

    # Keep weighted for comparison
    precision_weighted = precision_score(labels, predictions, average="weighted")
    recall_weighted = recall_score(labels, predictions, average="weighted")
    f1_weighted = f1_score(labels, predictions, average="weighted")

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
        "precision_macro": precision_macro,
        "precision_weighted": precision_weighted,
        "recall_macro": recall_macro,
        "recall_weighted": recall_weighted,
    }


class OversamplingTrainer(Trainer):
    """
    A custom Trainer class that implements weighted random sampling
    to handle class imbalance in the training dataset.
    """

    def _get_train_sampler(self, dataset):  # <-- FIX 1: Add 'dataset' argument
        # Get the labels from the training dataset
        labels = dataset["labels"]  # <-- FIX 2: Use the 'dataset' argument

        # Calculate class counts
        class_counts = np.bincount(labels)

        # Calculate weights for each class (inverse frequency)
        # We use 1.0 / count. Adding a small epsilon to avoid division by zero.
        class_weights = 1.0 / (class_counts + 1e-8)

        # Create a weight for each *sample* in the dataset
        # Every sample will get the weight corresponding to its class
        sample_weights = class_weights[labels]

        # Convert to a torch tensor
        sample_weights = torch.from_numpy(sample_weights).double()

        print(f"Oversampling enabled. Class counts: {class_counts}")
        print(f"Class weights: {class_weights}")

        # Create the WeightedRandomSampler
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True,  # 'replacement=True' is what enables oversampling
        )

        return sampler


# KEY CHANGES: Disable evaluation during training
training_args = TrainingArguments(
    output_dir="ModernBERT_QEvasion_model",
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="epoch",
    load_best_model_at_end=False,
    push_to_hub=False,
    logging_steps=100,
    report_to="none",
    fp16=True,  # Enable mixed precision (reduces memory usage)
    gradient_checkpointing=True,
)

# Trainer with ONLY training data
trainer = OversamplingTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],  # Only training data
    # eval_dataset is NOT provided here
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training (no evaluation during training)...")
trainer.train()

print("Training completed! Now evaluating on test data...")

# FINAL EVALUATION ON TEST DATA (one-time only)
test_results = trainer.evaluate(tokenized_datasets["test"])
print("\n" + "=" * 60)
print("FINAL TEST RESULTS (one-time evaluation)")
print("=" * 60)
for key, value in test_results.items():
    if key not in [
        "epoch",
        "eval_runtime",
        "eval_samples_per_second",
        "eval_steps_per_second",
    ]:
        print(f"{key}: {value:.4f}")

# Optional: Get detailed predictions
print("\nDetailed predictions analysis:")
predictions = trainer.predict(tokenized_datasets["test"])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

from sklearn.metrics import classification_report, confusion_matrix

print("\nClassification Report:")
print(
    classification_report(
        true_labels,
        predicted_labels,
        target_names=[id2label[i] for i in range(num_labels)],
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predicted_labels))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

Map:   0%|          | 0/308 [00:00<?, ? examples/s]

Dataset ready:
DatasetDict({
    train: Dataset({
        features: ['interview_answer', 'question', 'labels'],
        num_rows: 3448
    })
    test: Dataset({
        features: ['interview_answer', 'question', 'labels'],
        num_rows: 308
    })
})
Labels mapped: {'Clear Reply': 0, 'Ambivalent': 1, 'Clear Non-Reply': 2}


Map:   0%|          | 0/3448 [00:00<?, ? examples/s]

...

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-3470770585.py:143: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `OversamplingTrainer.__init__`. Use `processing_class` instead.
  trainer = OversamplingTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting training (no evaluation during training)...
Oversampling enabled. Class counts: [1052 2040  356]
Class weights: [0.00095057 0.0004902  0.00280899]


Step,Training Loss
100,0.994200
200,0.831400
300,0.733500
400,0.628500
500,0.570100
600,0.469300
700,0.480700
800,0.450800
900,0.406800
1000,0.401800


Training completed! Now evaluating on test data...



FINAL TEST RESULTS (one-time evaluation)
eval_loss: 1.3790
eval_accuracy: 0.7013
eval_f1_macro: 0.5874
eval_f1_weighted: 0.6977
eval_precision_macro: 0.6263
eval_precision_weighted: 0.6985
eval_recall_macro: 0.5670
eval_recall_weighted: 0.7013

Detailed predictions analysis:

Classification Report:
                 precision    recall  f1-score   support

    Clear Reply       0.53      0.56      0.54        79
     Ambivalent       0.78      0.80      0.79       206
Clear Non-Reply       0.57      0.35      0.43        23

       accuracy                           0.70       308
      macro avg       0.63      0.57      0.59       308
   weighted avg       0.70      0.70      0.70       308


Confusion Matrix:
[[ 44  33   2]
 [ 38 164   4]
 [  1  14   8]]
